In [33]:
import pandas as pd
from IPython.display import display
import requests
import os
from pathlib import Path
import sys
import json
from rapidfuzz import process, utils
from collections import Counter

In [2]:
# Getting parent directory
PARENT_DIR = str(Path.cwd().parent)
sys.path.insert(0, PARENT_DIR)
print("Added to sys.path:", PARENT_DIR)

Added to sys.path: /home/kamal/Desktop/UIDAI-Hackathon-2026


In [3]:
df = pd.read_csv("../data/raw/pincode.csv")
df.head()

,circlename,regionname,divisionname,officename,pincode,officetype,delivery,district,statename,latitude,longitude
0,Telangana Circle,Hyderabad Region,Adilabad Division,Kothimir B.O,504273,BO,Delivery,KUMURAM BHEEM ASIFABAD,TELANGANA,19.3638689,79.5376658
1,Telangana Circle,Hyderabad Region,Adilabad Division,Papanpet B.O,504299,BO,Delivery,KUMURAM BHEEM ASIFABAD,TELANGANA,19.4764899,79.5839230
2,Telangana Circle,Hyderabad Region,Adilabad Division,Kukuda B.O,504299,BO,Delivery,KUMURAM BHEEM ASIFABAD,TELANGANA,NaN,NaN
3,Telangana Circle,Hyderabad Region,Adilabad Division,Bareguda B.O,504296,BO,Delivery,KUMURAM BHEEM ASIFABAD,TELANGANA,19.3285752,79.4760132
4,Telangana Circle,Hyderabad Region,Adilabad Division,Mosam B.O,504296,BO,Delivery,KUMURAM BHEEM ASIFABAD,TELANGANA,19.3778044,79.6165209


In [4]:
df.shape

(165627, 11)

In [5]:
df.duplicated().sum()

np.int64(2)

In [6]:
# Remove duplicates
df = df.drop_duplicates().reset_index(drop=True)
df.shape

(165625, 11)

In [7]:
# check null values
df.isnull().sum()

circlename          0
regionname        315
divisionname        0
officename          0
pincode             0
officetype          0
delivery            0
district          715
statename         715
latitude        12007
longitude       12002
dtype: int64

In [8]:
# checking duplicates with statename, district, pincode
df[df.duplicated(subset=["statename", "district", "pincode"])]

,circlename,regionname,divisionname,officename,pincode,officetype,delivery,district,statename,latitude,longitude
2,Telangana Circle,Hyderabad Region,Adilabad Division,Kukuda B.O,504299,BO,Delivery,KUMURAM BHEEM ASIFABAD,TELANGANA,NaN,NaN
4,Telangana Circle,Hyderabad Region,Adilabad Division,Mosam B.O,504296,BO,Delivery,KUMURAM BHEEM ASIFABAD,TELANGANA,19.3778044,79.6165209
5,Telangana Circle,Hyderabad Region,Adilabad Division,Penchikalpet B.O,504296,BO,Delivery,KUMURAM BHEEM ASIFABAD,TELANGANA,19.2402360,79.8205696
7,Telangana Circle,Hyderabad Region,Adilabad Division,Old Town B.O,504296,BO,Non Delivery,KUMURAM BHEEM ASIFABAD,TELANGANA,19.3454350,79.4782916
8,Telangana Circle,Hyderabad Region,Adilabad Division,Kammergaon B.O,504296,BO,Delivery,KUMURAM BHEEM ASIFABAD,TELANGANA,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
165618,West Bengal Circle,South Bengal Region,Tamluk Division,Narghat SO,721633,PO,Delivery,MEDINIPUR EAST,WEST BENGAL,22.0758000,87.5322000
165619,West Bengal Circle,South Bengal Region,Tamluk Division,Panskura RS SO,721152,PO,Delivery,MEDINIPUR EAST,WEST BENGAL,22.3930278,87.7428056
165620,West Bengal Circle,South Bengal Region,Tamluk Division,Raghunathbari SO,721634,PO,Delivery,MEDINIPUR EAST,WEST BENGAL,22.3536629,87.7976060
165621,West Bengal Circle,South Bengal Region,Tamluk Division,Ramchandrapur SO East Midnapore,721647,PO,Delivery,MEDINIPUR EAST,WEST BENGAL,22.2975,87.7542


In [9]:
# using only those columns and removing duplicates
df = (
    df[["statename", "district", "pincode"]]
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)

In [10]:
df.shape

(21162, 3)

In [11]:
# check unique pincodes
df["pincode"].nunique()

19586

In [12]:
# cheking unique states
df["statename"].unique(), df["statename"].nunique()

(array(['TELANGANA', 'ANDHRA PRADESH', 'ASSAM', nan, 'BIHAR',
        'CHHATTISGARH', 'MEGHALAYA', 'JHARKHAND', 'KARNATAKA',
        'HIMACHAL PRADESH', 'JAMMU AND KASHMIR', 'LADAKH', 'GUJARAT',
        'THE DADRA AND NAGAR HAVELI AND DAMAN AND DIU', 'HARYANA', 'DELHI',
        'MAHARASHTRA', 'KERALA', 'MANIPUR', 'MIZORAM', 'NAGALAND',
        'PUDUCHERRY', 'MADHYA PRADESH', 'GOA', 'PUNJAB', 'RAJASTHAN',
        'ODISHA', 'TRIPURA', 'ARUNACHAL PRADESH', 'TAMIL NADU',
        'CHANDIGARH', 'UTTAR PRADESH', 'UTTARAKHAND', 'WEST BENGAL',
        'ANDAMAN AND NICOBAR ISLANDS', 'SIKKIM', 'LAKSHADWEEP'],
       dtype=object),
 36)

In [13]:
# check null values
df.isnull().sum()

statename    338
district     338
pincode        0
dtype: int64

In [14]:
df[df[["statename", "district"]].isnull().all(axis=1)]

,statename,district,pincode
691,NaN,NaN,523261
772,NaN,NaN,521250
863,NaN,NaN,811315
884,NaN,NaN,811106
921,NaN,NaN,805108
...,...,...,...
21008,NaN,NaN,411075
21017,NaN,NaN,799001
21072,NaN,NaN,600202
21143,NaN,NaN,736209


In [15]:
df[df[["statename", "district"]].isnull().any(axis=1)]

,statename,district,pincode
691,NaN,NaN,523261
772,NaN,NaN,521250
863,NaN,NaN,811315
884,NaN,NaN,811106
921,NaN,NaN,805108
...,...,...,...
21008,NaN,NaN,411075
21017,NaN,NaN,799001
21072,NaN,NaN,600202
21143,NaN,NaN,736209


In [16]:
# no of unique pincodes
(
    df["pincode"].nunique(),
    df.dropna(subset=["statename", "district"], how="all")["pincode"].nunique(),
)

(19586, 19486)

In [17]:
df.dropna(subset=["statename", "district"])

,statename,district,pincode
0,TELANGANA,KUMURAM BHEEM ASIFABAD,504273
1,TELANGANA,KUMURAM BHEEM ASIFABAD,504299
2,TELANGANA,KUMURAM BHEEM ASIFABAD,504296
3,TELANGANA,MANCHERIAL,504209
4,TELANGANA,MANCHERIAL,504272
...,...,...,...
21156,WEST BENGAL,MEDINIPUR WEST,721303
21157,WEST BENGAL,MEDINIPUR EAST,721606
21158,WEST BENGAL,MEDINIPUR EAST,721604
21159,WEST BENGAL,MEDINIPUR EAST,721171


In [18]:
# null pincodes
null_pincodes = df[df[["statename", "district"]].isnull().any(axis=1)][
    "pincode"
].unique()

In [19]:
null_pincodes[0]

np.int64(523261)

In [48]:
def get_pincode_info(pincode: int) -> dict[str, str]:
    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "*/*",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive",
    }
    url = f"https://api.postalpincode.in/pincode/{pincode}"
    response = requests.get(url, headers=headers, timeout=10)
    if response.status_code == 200:
        data = response.json()[0]
        if data and data["Status"] == "Success":
            states, districts = set(), set()
            for each_data in data["PostOffice"]:
                states.add(each_data["State"])
                districts.add(each_data["District"])

            return {
                "state": list(states),
                "district": list(districts),
            }
        else:
            print(f"No data found for pincode {pincode}")
            return {}

    else:
        print(f"{response.status_code}: Failed to fetch data for pincode {pincode}")
        return {}

In [49]:
get_pincode_info(null_pincodes[0])

{'state': ['Andhra Pradesh'], 'district': ['Prakasam']}

In [22]:
# will store all pincode:{state:, district:} info for all available pincodes
cleand_df = df.dropna(subset=["statename", "district"], how="all").copy()
pincode_info = {}

In [23]:
cleand_df.duplicated().sum()

np.int64(0)

In [24]:
cleand_df

,statename,district,pincode
0,TELANGANA,KUMURAM BHEEM ASIFABAD,504273
1,TELANGANA,KUMURAM BHEEM ASIFABAD,504299
2,TELANGANA,KUMURAM BHEEM ASIFABAD,504296
3,TELANGANA,MANCHERIAL,504209
4,TELANGANA,MANCHERIAL,504272
...,...,...,...
21156,WEST BENGAL,MEDINIPUR WEST,721303
21157,WEST BENGAL,MEDINIPUR EAST,721606
21158,WEST BENGAL,MEDINIPUR EAST,721604
21159,WEST BENGAL,MEDINIPUR EAST,721171


normalizing it

In [25]:
official_states = json.load(open(f"{PARENT_DIR}/data/states.json", "r"))[
    "official_states"
]

In [26]:
def normalize_text(s):
    return s.lower().strip().replace("&", "and").replace(" ", "")


def get_closest_match(key: str, ground: list[str], score_cutoff: int = 80) -> str:
    """Give me the doubttful key and I will return the closest match from the grounding list using fuzzy matching.

    Args:
        key (str): the value to be matched
        ground (list[str]): the list of possible matches
        score_cutoff (int): minimum score to consider a match. Defaults to 80.

    Returns:
        str: the closest match if found, else the original key
    """
    # Returns the best match from official_states if score > 80%
    match = process.extractOne(key, ground, score_cutoff=score_cutoff)
    return match[0] if match else key

In [27]:
def state_mapping(df):
    # Get unique dirty states
    df["statename"] = df["statename"].astype(str).apply(normalize_text)

    dirty_states = df["statename"].unique()

    # Create and apply an automated mapping dictionary
    auto_mapping = {ds: get_closest_match(ds, official_states) for ds in dirty_states}
    df["statename"] = df["statename"].map(auto_mapping)

    return df

In [28]:
state_mapping(cleand_df)["statename"].unique()

array(['telangana', 'andhrapradesh', 'assam', 'bihar', 'chhattisgarh',
       'meghalaya', 'jharkhand', 'karnataka', 'himachalpradesh',
       'jammuandkashmir', 'ladakh', 'gujarat',
       'dadraandnagarhavelianddamananddiu', 'haryana', 'delhi',
       'maharashtra', 'kerala', 'manipur', 'mizoram', 'nagaland',
       'puducherry', 'madhyapradesh', 'goa', 'punjab', 'rajasthan',
       'odisha', 'tripura', 'arunachalpradesh', 'tamilnadu', 'chandigarh',
       'uttarpradesh', 'uttarakhand', 'westbengal',
       'andamanandnicobarislands', 'sikkim', 'lakshadweep'], dtype=object)

In [30]:
state_mapping(cleand_df)

,statename,district,pincode
0,telangana,KUMURAM BHEEM ASIFABAD,504273
1,telangana,KUMURAM BHEEM ASIFABAD,504299
2,telangana,KUMURAM BHEEM ASIFABAD,504296
3,telangana,MANCHERIAL,504209
4,telangana,MANCHERIAL,504272
...,...,...,...
21156,westbengal,MEDINIPUR WEST,721303
21157,westbengal,MEDINIPUR EAST,721606
21158,westbengal,MEDINIPUR EAST,721604
21159,westbengal,MEDINIPUR EAST,721171


In [31]:
cleand_df = state_mapping(cleand_df).copy()

In [32]:
# do duplicate pincodes in our data
cleand_df.shape[0], cleand_df["pincode"].nunique()

(20824, 19486)

In [35]:
# checking what are they
temp_ = df["pincode"].value_counts().reset_index()
temp_

,pincode,count
0,192124,4
1,791102,4
2,791001,4
3,796410,4
4,795146,4
...,...,...
19581,143528,1
19582,143527,1
19583,143526,1
19584,143519,1


In [38]:
cleand_df[cleand_df["pincode"] == temp_.iloc[0, 0]]

,statename,district,pincode
3304,jammuandkashmir,ANANTNAG,192124
3305,jammuandkashmir,SHOPIAN,192124
15059,jammuandkashmir,PULWAMA,192124
18092,jammuandkashmir,KULGAM,192124


In [44]:
# so pincodes can have different districts but not different states
# will check if a pincode having more than one occurance but must be same state
temp_ = cleand_df[["statename", "pincode"]].drop_duplicates()
temp_ = temp_["pincode"].value_counts().reset_index()
dup_pincodes = temp_[temp_["count"] > 1]["pincode"].values
dup_pincodes

array([396193, 504214, 504215, 531149, 247662, 813206, 605105, 802131,
       605007, 605502, 607402, 605106, 505187, 605501, 605107, 343027,
       607403, 505525, 335513, 311601, 110025, 503188, 332028, 503145,
       609604, 509324, 605102, 335526, 782410, 533464, 333022, 605110,
       244924, 506003, 756048, 244923, 305402, 503235, 503321, 781131,
       505325, 503230, 605111, 396230, 503225, 396235, 396215, 509132,
       609603, 605014, 781029, 396240])

In [50]:
# checking with api
get_pincode_info(396193)

{'state': ['Dadra & Nagar Haveli', 'Gujarat'],
 'district': ['Dadra & Nagar Haveli', 'Valsad']}

In [ ]:
# appending with

In [ ]:
df["pincode"].value_counts().reset_index()

,pincode,count
0,192124,4
1,791102,4
2,791001,4
3,796410,4
4,795146,4
...,...,...
19581,143528,1
19582,143527,1
19583,143526,1
19584,143519,1


--------

In [ ]:
result = (
    df.set_index(["pincode", "statename"])["district"].unstack().to_dict(orient="index")
)

ValueError: Index contains duplicate entries, cannot reshape

In [ ]:
# testing
